 Environment setup

In [1]:
import subprocess
import sys
from importlib.metadata import version, PackageNotFoundError

PINNED = {
    "transformers": "4.41.2",
    "sentence-transformers": "2.7.0",
    "accelerate": "0.31.0",
    "rank-bm25": "0.2.2",
}

OPTIONAL = {
    "faiss-cpu": "1.8.0.post1",
}

def needs_install(pkg, wanted):
    try:
        return version(pkg) != wanted
    except PackageNotFoundError:
        return True

required_specs = [f"{pkg}=={ver}" for pkg, ver in PINNED.items() if needs_install(pkg, ver)]
optional_specs = [f"{pkg}=={ver}" for pkg, ver in OPTIONAL.items() if needs_install(pkg, ver)]

if required_specs:
    print("Installing / aligning required packages:")
    for spec in required_specs:
        print("  -", spec)
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + required_specs)
    except Exception as e:
        print("Package installation warning:", e)
        print("these versions install correctly.")
else:
    print("Required package versions already aligned.")

if optional_specs:
    try:
        print("Installing optional packages:")
        for spec in optional_specs:
            print("  -", spec)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + optional_specs)
    except Exception as e:
        print("Optional FAISS install failed", e)
else:
    print("Optional package versions already aligned.")


Installing / aligning required packages:
  - transformers==4.41.2
  - sentence-transformers==2.7.0
  - accelerate==0.31.0
  - rank-bm25==0.2.2
Installing optional packages:
  - faiss-cpu==1.8.0.post1


In [ ]:

import os
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive"
else:
    DRIVE_ROOT = os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive")
    print("Colab not detected. Using DRIVE_ROOT =", DRIVE_ROOT)

PAN_DRIVE_DIR = f"{DRIVE_ROOT}"


PAN_TRAIN_ZIP = f"{PAN_DRIVE_DIR}/pan25-generated-plagiarism-detection-train.zip"
PAN_VAL_ZIP = f"{PAN_DRIVE_DIR}/pan25-generated-plagiarism-detection-validation.zip"

QQP_LOCAL_OUTER_ZIP = "/content/quora-question-pairs.zip"
QQP_LOCAL_INNER_ZIP = "/content/train.csv.zip"
QQP_LOCAL_CSV = "/content/train.csv"



print("Configured paths")
print("-" * 80)
print("PAN train zip       :", PAN_TRAIN_ZIP, "| exists:", Path(PAN_TRAIN_ZIP).exists())
print("PAN validation zip  :", PAN_VAL_ZIP, "| exists:", Path(PAN_VAL_ZIP).exists())
print("QQP local outer zip :", QQP_LOCAL_OUTER_ZIP, "| exists:", Path(QQP_LOCAL_OUTER_ZIP).exists())



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Configured paths
--------------------------------------------------------------------------------
PAN train zip       : /content/drive/MyDrive/pan25-generated-plagiarism-detection-train.zip | exists: True
PAN validation zip  : /content/drive/MyDrive/pan25-generated-plagiarism-detection-validation.zip | exists: True
QQP local outer zip : /content/quora-question-pairs.zip | exists: True


In [ ]:

RUNTIME_PROFILE = "full"
RUN_BASELINES = True
RUN_BI_ENCODER = True
RUN_CROSS_ENCODER = True
RUN_FINAL_EVAL = True
RUN_ABLATIONS = True

# Checkpoint
ENABLE_STAGE_CHECKPOINTS = True
AUTO_RESUME_FROM_CHECKPOINTS = True
AUTO_BUILD_CHECKPOINT_ZIPS = True
AUTO_DOWNLOAD_STAGE_ZIPS = False
FORCE_RERUN_STAGES = []
DOWNLOAD_ALL_STAGE_ZIPS_AT_END = False

print("Runtime profile            :", RUNTIME_PROFILE)
print("Run baselines              :", RUN_BASELINES)
print("Run bi-encoder             :", RUN_BI_ENCODER)
print("Run cross-encoder          :", RUN_CROSS_ENCODER)
print("Run final eval             :", RUN_FINAL_EVAL)
print("Run ablations              :", RUN_ABLATIONS)
print("Enable stage checkpoints   :", ENABLE_STAGE_CHECKPOINTS)
print("Auto resume checkpoints    :", AUTO_RESUME_FROM_CHECKPOINTS)
print("Auto build checkpoint zips :", AUTO_BUILD_CHECKPOINT_ZIPS)
print("Auto download stage zips   :", AUTO_DOWNLOAD_STAGE_ZIPS)
print("Force rerun stages         :", FORCE_RERUN_STAGES)


Runtime profile            : full
Run baselines              : True
Run bi-encoder             : True
Run cross-encoder          : True
Run final eval             : True
Run ablations              : True
Enable stage checkpoints   : True
Auto resume checkpoints    : True
Auto build checkpoint zips : True
Auto download stage zips   : False
Force rerun stages         : []


Imports and configuration

In [ ]:
from __future__ import annotations

import os
import io
import gc
import math
import json
import time
import copy
import random
import zipfile
import hashlib
import pickle
import warnings
import shutil
import unicodedata
import importlib.metadata
import xml.etree.ElementTree as ET

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from scipy import sparse

import torch
from torch.utils.data import DataLoader

from transformers import AutoTokenizer
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample, losses

try:
    import faiss
    HAS_FAISS = True
except Exception:
    faiss = None
    HAS_FAISS = False

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

def get_version(pkg_name: str):
    try:
        return importlib.metadata.version(pkg_name)
    except Exception:
        return "not-found"

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

@dataclass
class ExperimentConfig:
    pan_train_zip: str | None = PAN_TRAIN_ZIP
    pan_val_zip: str | None = PAN_VAL_ZIP

    qqp_local_outer_zip: str | None = QQP_LOCAL_OUTER_ZIP
    qqp_local_inner_zip: str | None = QQP_LOCAL_INNER_ZIP
    qqp_local_csv: str | None = QQP_LOCAL_CSV


    work_dir: str = "./bece_pan25_artifacts"
    cache_dir: str = "./bece_pan25_artifacts/cache"
    model_dir: str = "./bece_pan25_artifacts/models"
    report_dir: str = "./bece_pan25_artifacts/reports"
    xml_out_dir: str = "./bece_pan25_artifacts/xml_outputs"
    checkpoint_dir: str = "./bece_pan25_artifacts/checkpoints"
    checkpoint_bundle_dir: str = "./bece_pan25_artifacts/checkpoint_bundles"

    tokenizer_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    bi_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    sbert_baseline_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    ce_model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"

    chunk_tokens: int = 192
    chunk_stride: int = 96
    min_overlap_chars: int = 32

    lowercase: bool = False
    remove_punctuation: bool = False
    collapse_whitespace: bool = True

    top_k: int = 10
    hard_negative_k: int = 20
    ce_threshold: float = 0.50
    merge_gap_chars: int = 60
    use_sentence_boundary_expansion: bool = True

    seed: int = 42
    qqp_max_rows: int | None = None
    pan_max_train_pairs: int | None = None
    pan_max_val_pairs: int | None = None

    pan_dev_fraction: float = 0.10
    allow_single_archive_dev_mode: bool = False

    qqp_epochs: int = 1
    pan_bi_epochs: int = 1
    ce_epochs: int = 1
    bi_batch_size: int = 32
    ce_batch_size: int = 16
    embedding_batch_size: int = 64
    learning_rate: float = 2e-5

    use_hard_negatives: bool = True
    use_altered_negatives: bool = True

    dry_run: bool = False
    dry_run_pairs: int = 64
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    save_models: bool = True
    save_reports: bool = True
    save_checkpoints: bool = ENABLE_STAGE_CHECKPOINTS
    auto_resume_checkpoints: bool = AUTO_RESUME_FROM_CHECKPOINTS
    auto_build_checkpoint_zips: bool = AUTO_BUILD_CHECKPOINT_ZIPS
    auto_download_stage_zips: bool = AUTO_DOWNLOAD_STAGE_ZIPS

    threshold_grid: tuple = (0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70)

CFG = ExperimentConfig()

if RUNTIME_PROFILE == "colab_safe":
    CFG.bi_batch_size = 16
    CFG.ce_batch_size = 8
    CFG.embedding_batch_size = 32
    CFG.top_k = 8
    CFG.hard_negative_k = 12
    CFG.merge_gap_chars = 40
else:
    CFG.bi_batch_size = 32
    CFG.ce_batch_size = 16
    CFG.embedding_batch_size = 64
    CFG.top_k = 10
    CFG.hard_negative_k = 20
    CFG.merge_gap_chars = 60

set_seed(CFG.seed)

for p in [CFG.work_dir, CFG.cache_dir, CFG.model_dir, CFG.report_dir, CFG.xml_out_dir, CFG.checkpoint_dir, CFG.checkpoint_bundle_dir]:
    Path(p).mkdir(parents=True, exist_ok=True)

print("Versions")
print("-" * 80)
for pkg in ["numpy", "pandas", "scikit-learn", "torch", "transformers", "sentence-transformers", "rank-bm25"]:
    print(f"{pkg:24s}: {get_version(pkg)}")
print("-" * 80)
print("Device:", CFG.device)
print("FAISS available:", HAS_FAISS)
print("Working directory:", Path(CFG.work_dir).resolve())
print("Configured PAN train zip:", CFG.pan_train_zip)
print("Configured PAN val zip:", CFG.pan_val_zip)


Versions
--------------------------------------------------------------------------------
numpy                   : 1.26.4
pandas                  : 2.2.2
scikit-learn            : 1.6.1
torch                   : 2.9.0+cpu
transformers            : 4.41.2
sentence-transformers   : 2.7.0
rank-bm25               : 0.2.2
--------------------------------------------------------------------------------
Device: cpu
FAISS available: True
Working directory: /content/bece_pan25_artifacts
Configured PAN train zip: /content/drive/MyDrive/pan25-generated-plagiarism-detection-train.zip
Configured PAN val zip: /content/drive/MyDrive/pan25-generated-plagiarism-detection-validation.zip


General utilities

In [ ]:

def save_json(obj, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def interval_overlap(a_start: int, a_end: int, b_start: int, b_end: int) -> int:
    return max(0, min(a_end, b_end) - max(a_start, b_start))

def safe_div(n: float, d: float) -> float:
    return float(n) / float(d) if d else 0.0

def cosine_search_numpy(query_vecs: np.ndarray, doc_vecs: np.ndarray, top_k: int = 10):
    if len(query_vecs) == 0 or len(doc_vecs) == 0:
        return np.zeros((len(query_vecs), 0), dtype=np.float32), np.zeros((len(query_vecs), 0), dtype=int)
    q = query_vecs / np.clip(np.linalg.norm(query_vecs, axis=1, keepdims=True), 1e-12, None)
    d = doc_vecs / np.clip(np.linalg.norm(doc_vecs, axis=1, keepdims=True), 1e-12, None)
    sims = q @ d.T
    idx = np.argsort(-sims, axis=1)[:, :top_k]
    scores = np.take_along_axis(sims, idx, axis=1)
    return scores, idx

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def print_header(title: str):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

def maybe_save_df(df: pd.DataFrame, path: str | Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_pickle(obj, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def load_pickle(path: str | Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def stage_dir(stage_name: str) -> Path:
    return Path(CFG.checkpoint_dir) / stage_name


def stage_manifest_path(stage_name: str) -> Path:
    return stage_dir(stage_name) / "manifest.json"


def stage_bundle_path(stage_name: str) -> Path:
    return Path(CFG.checkpoint_bundle_dir) / f"{stage_name}.zip"


def should_resume_stage(stage_name: str) -> bool:
    return bool(
        CFG.auto_resume_checkpoints
        and CFG.save_checkpoints
        and stage_manifest_path(stage_name).exists()
        and stage_name not in set(FORCE_RERUN_STAGES)
    )


def stage_checkpoint_exists(stage_name: str) -> bool:
    return stage_manifest_path(stage_name).exists()


def _write_df(df: pd.DataFrame, path: Path) -> str:
    df.to_pickle(path)
    return path.name


def _write_jsonable(obj, path: Path) -> str:
    save_json(obj, path)
    return path.name


def _copy_path(src_path: str | Path, dst_path: Path):
    src_path = Path(src_path)
    if not src_path.exists():
        return None
    if src_path.is_dir():
        if dst_path.exists():
            shutil.rmtree(dst_path)
        shutil.copytree(src_path, dst_path)
    else:
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_path, dst_path)
    return dst_path.name


def save_stage_checkpoint(stage_name: str, payload: dict | None = None, extra_paths: dict | None = None, metadata: dict | None = None):
    payload = payload or {}
    extra_paths = extra_paths or {}
    metadata = metadata or {}

    sdir = stage_dir(stage_name)
    if sdir.exists():
        shutil.rmtree(sdir)
    sdir.mkdir(parents=True, exist_ok=True)

    payload_files = {}
    for key, value in payload.items():
        if isinstance(value, pd.DataFrame):
            payload_files[key] = _write_df(value, sdir / f"{key}.pkl")
        elif isinstance(value, (dict, list, tuple, str, int, float, bool)) or value is None:
            payload_files[key] = _write_jsonable(value, sdir / f"{key}.json")
        else:
            payload_files[key] = f"{key}.pkl"
            save_pickle(value, sdir / payload_files[key])

    copied_paths = {}
    for key, src in extra_paths.items():
        if src is None:
            continue
        src_path = Path(src)
        dst_name = src_path.name
        dst_path = sdir / dst_name
        copied = _copy_path(src_path, dst_path)
        if copied is not None:
            copied_paths[key] = copied

    manifest = {
        "stage_name": stage_name,
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "payload_files": payload_files,
        "copied_paths": copied_paths,
        "metadata": metadata,
    }
    save_json(manifest, stage_manifest_path(stage_name))
    print(f"Checkpoint saved for {stage_name}: {stage_manifest_path(stage_name)}")

    bundle = None
    if CFG.auto_build_checkpoint_zips:
        bundle = build_stage_bundle(stage_name)
    if CFG.auto_download_stage_zips and bundle is not None:
        download_file_from_notebook(bundle)
    return manifest


def load_stage_checkpoint(stage_name: str):
    manifest_path = stage_manifest_path(stage_name)
    if not manifest_path.exists():
        raise FileNotFoundError(f"Checkpoint not found for stage: {stage_name}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    sdir = stage_dir(stage_name)

    loaded = {"manifest": manifest}
    for key, fname in manifest.get("payload_files", {}).items():
        fpath = sdir / fname
        if fname.endswith('.json'):
            loaded[key] = json.loads(fpath.read_text(encoding='utf-8'))
        elif fname.endswith('.pkl'):
            loaded[key] = load_pickle(fpath)
        else:
            loaded[key] = fpath
    for key, fname in manifest.get("copied_paths", {}).items():
        loaded[key] = sdir / fname
    return loaded


def build_stage_bundle(stage_name: str):
    sdir = stage_dir(stage_name)
    if not sdir.exists():
        print(f"No checkpoint directory to bundle for stage: {stage_name}")
        return None
    bundle_path = stage_bundle_path(stage_name)
    bundle_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file_path in sdir.rglob('*'):
            if file_path.is_file():
                zf.write(file_path, arcname=file_path.relative_to(sdir.parent))
    print(f"Checkpoint bundle created: {bundle_path}")
    return bundle_path


def list_available_checkpoints():
    rows = []
    root = Path(CFG.checkpoint_dir)
    for manifest_path in sorted(root.glob('*/manifest.json')):
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        stage_name = manifest.get('stage_name', manifest_path.parent.name)
        rows.append({
            'stage_name': stage_name,
            'created_at': manifest.get('created_at'),
            'payload_keys': ', '.join(sorted(manifest.get('payload_files', {}).keys())),
            'copied_paths': ', '.join(sorted(manifest.get('copied_paths', {}).keys())),
            'bundle_exists': stage_bundle_path(stage_name).exists(),
            'bundle_path': str(stage_bundle_path(stage_name)),
        })
    return pd.DataFrame(rows)


def download_file_from_notebook(path: str | Path):
    path = Path(path)
    print(f"Download-ready file: {path}")
    if 'google.colab' in sys.modules:
        try:
            from google.colab import files
            files.download(str(path))
        except Exception as e:
            print('Download warning:', e)


def download_stage_checkpoint(stage_name: str):
    bundle = stage_bundle_path(stage_name)
    if not bundle.exists():
        bundle = build_stage_bundle(stage_name)
    if bundle is not None:
        download_file_from_notebook(bundle)


def download_all_stage_checkpoints():
    df = list_available_checkpoints()
    if df.empty:
        print('No stage checkpoints found.')
        return
    for stage_name in df['stage_name'].tolist():
        download_stage_checkpoint(stage_name)


In [ ]:

def discover_pan_layout(zip_path: str) -> dict:
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()

    pairs_candidates = [n for n in names if n.endswith("/pairs") or n.endswith("pairs")]
    if not pairs_candidates:
        raise FileNotFoundError(f"No PAN pairs file found inside {zip_path}")

    pairs_path = sorted(pairs_candidates, key=len)[0]
    text_root = pairs_path.rsplit("/", 1)[0]

    truth_candidates = [n for n in names if n.endswith(".xml")]
    if not truth_candidates:
        raise FileNotFoundError(f"No truth XML directory found inside {zip_path}")
    truth_root = sorted({"/".join(n.split("/")[:-1]) for n in truth_candidates}, key=len)[0]

    return {
        "zip_path": zip_path,
        "pairs_path": pairs_path,
        "text_root": text_root,
        "truth_root": truth_root,
    }


def parse_pan_truth_xml(xml_bytes: bytes) -> dict:
    root = ET.fromstring(xml_bytes)
    doc_ref = root.attrib.get("reference")
    plagiarism = []
    altered = []
    meta = {}
    for feat in root.findall("feature"):
        name = feat.attrib.get("name", "")
        attrs = dict(feat.attrib)
        if name == "plagiarism":
            plagiarism.append({
                "this_offset": int(attrs.get("this_offset", 0)),
                "this_length": int(attrs.get("this_length", 0)),
                "this_end": int(attrs.get("this_offset", 0)) + int(attrs.get("this_length", 0)),
                "source_reference": attrs.get("source_reference"),
                "source_offset": int(attrs.get("source_offset", 0)),
                "source_length": int(attrs.get("source_length", 0)),
                "source_end": int(attrs.get("source_offset", 0)) + int(attrs.get("source_length", 0)),
                "llm": attrs.get("llm"),
                "type": attrs.get("type"),
                "obfuscation": attrs.get("obfuscation"),
            })
        elif name == "altered":
            altered.append({
                "this_offset": int(attrs.get("this_offset", 0)),
                "this_length": int(attrs.get("this_length", 0)),
                "this_end": int(attrs.get("this_offset", 0)) + int(attrs.get("this_length", 0)),
                "llm": attrs.get("llm"),
                "type": attrs.get("type"),
            })
        else:
            meta.setdefault(name, []).append(attrs)
    return {
        "document_reference": doc_ref,
        "plagiarism": plagiarism,
        "altered": altered,
        "meta": meta,
    }


def read_text_from_zip(zf: zipfile.ZipFile, inner_path: str) -> str:
    return zf.read(inner_path).decode("utf-8", errors="ignore")


def build_truth_xml_name(susp_name: str, src_name: str) -> str:
    susp_stem = Path(susp_name).stem
    src_stem = Path(src_name).stem
    return f"{susp_stem}-{src_stem}.xml"


def load_pan_records_from_zip(zip_path: str, split_name: str, max_pairs: int | None = None) -> list[dict]:
    layout = discover_pan_layout(zip_path)
    records = []

    with zipfile.ZipFile(zip_path) as zf:
        pairs_lines = read_text_from_zip(zf, layout["pairs_path"]).splitlines()
        pairs = [line.strip().split() for line in pairs_lines if line.strip()]

        if max_pairs is not None:
            pairs = pairs[:max_pairs]

        nameset = set(zf.namelist())
        for susp_name, src_name in tqdm(pairs, desc=f"Loading PAN {split_name}"):
            xml_name = build_truth_xml_name(susp_name, src_name)
            xml_path = f'{layout["truth_root"]}/{xml_name}'
            susp_path = f'{layout["text_root"]}/susp/{susp_name}'
            src_path = f'{layout["text_root"]}/src/{src_name}'

            truth = parse_pan_truth_xml(zf.read(xml_path)) if xml_path in nameset else {
                "document_reference": susp_name,
                "plagiarism": [],
                "altered": [],
                "meta": {},
            }

            susp_text = read_text_from_zip(zf, susp_path)
            src_text = read_text_from_zip(zf, src_path)

            pair_id = f"{Path(susp_name).stem}__{Path(src_name).stem}"
            records.append({
                "pair_id": pair_id,
                "split": split_name,
                "zip_path": zip_path,
                "susp_name": susp_name,
                "src_name": src_name,
                "susp_text": susp_text,
                "src_text": src_text,
                "plagiarism_spans": truth["plagiarism"],
                "altered_spans": truth["altered"],
                "num_plagiarism_spans": len(truth["plagiarism"]),
                "num_altered_spans": len(truth["altered"]),
            })
    return records

Dataset loading

In [ ]:

def split_train_into_fit_dev(records: list[dict], dev_fraction: float = 0.10, seed: int = 42):
    pair_ids = [r["pair_id"] for r in records]
    fit_ids, dev_ids = train_test_split(
        pair_ids,
        test_size=dev_fraction,
        random_state=seed,
        shuffle=True,
    )
    fit_ids, dev_ids = set(fit_ids), set(dev_ids)
    fit_records = [r for r in records if r["pair_id"] in fit_ids]
    dev_records = [r for r in records if r["pair_id"] in dev_ids]
    return fit_records, dev_records


def prepare_pan_fit_dev_test_records(cfg: ExperimentConfig):
    train_exists = cfg.pan_train_zip and Path(cfg.pan_train_zip).exists()
    val_exists = cfg.pan_val_zip and Path(cfg.pan_val_zip).exists()

    if train_exists and val_exists:
        official_train_records = load_pan_records_from_zip(cfg.pan_train_zip, "train", cfg.pan_max_train_pairs)
        official_val_records = load_pan_records_from_zip(cfg.pan_val_zip, "validation", cfg.pan_max_val_pairs)
        fit_records, dev_records = split_train_into_fit_dev(
            official_train_records,
            dev_fraction=cfg.pan_dev_fraction,
            seed=cfg.seed,
        )
        test_records = official_val_records
        mode = "official_train_fit_dev_plus_official_validation_test"
        return fit_records, dev_records, test_records, mode

    if cfg.allow_single_archive_dev_mode:
        available = [p for p in [cfg.pan_train_zip, cfg.pan_val_zip] if p and Path(p).exists()]
        if not available:
            raise FileNotFoundError("No PAN archive found. Please set cfg.pan_train_zip and cfg.pan_val_zip.")
        only_zip = available[0]
        all_records = load_pan_records_from_zip(only_zip, "single_archive", cfg.pan_max_train_pairs)
        fit_records, dev_records = split_train_into_fit_dev(
            all_records,
            dev_fraction=cfg.pan_dev_fraction,
            seed=cfg.seed,
        )
        test_records = dev_records
        mode = "single_archive_dev_mode"
        return fit_records, dev_records, test_records, mode

    raise FileNotFoundError(
        "Both PAN train and validation ZIPs are required for the full notebook. "
        "Place them in Google Drive and keep the Drive-path cell correct."
    )


fit_records, dev_records, test_records, pan_mode = prepare_pan_fit_dev_test_records(CFG)
print("PAN mode:", pan_mode)
print("Fit pairs:", len(fit_records))
print("Dev pairs:", len(dev_records))
print("Test pairs:", len(test_records))


print('Sample fit pair IDs:', [r['pair_id'] for r in fit_records[:3]])
print('Sample dev pair IDs:', [r['pair_id'] for r in dev_records[:3]])
print('Sample test pair IDs:', [r['pair_id'] for r in test_records[:3]])


Loading PAN train:   0%|          | 0/62160 [00:00<?, ?it/s]

Loading PAN validation:   0%|          | 0/7976 [00:00<?, ?it/s]

PAN mode: official_train_fit_dev_plus_official_validation_test
Fit pairs: 55944
Dev pairs: 6216
Test pairs: 7976
Sample fit pair IDs: ['suspicious-document020468__source-document020468', 'suspicious-document020469__source-document020469', 'suspicious-document020470__source-document020470']
Sample dev pair IDs: ['suspicious-document020476__source-document020476', 'suspicious-document020523__source-document020523', 'suspicious-document020524__source-document020524']
Sample test pair IDs: ['suspicious-document010237__source-document010237', 'suspicious-document010240__source-document010240', 'suspicious-document010241__source-document010241']


In [ ]:

def summarize_pan_records(records: list[dict], split_name: str) -> pd.DataFrame:
    rows = []
    for r in records:
        for p in r["plagiarism_spans"]:
            rows.append({
                "split": split_name,
                "pair_id": r["pair_id"],
                "span_type": "plagiarism",
                "llm": p.get("llm"),
                "obfuscation": p.get("obfuscation"),
                "this_length": p.get("this_length"),
                "source_length": p.get("source_length"),
            })
        for a in r["altered_spans"]:
            rows.append({
                "split": split_name,
                "pair_id": r["pair_id"],
                "span_type": "altered",
                "llm": a.get("llm"),
                "obfuscation": None,
                "this_length": a.get("this_length"),
                "source_length": None,
            })
    return pd.DataFrame(rows)


pan_fit_summary = summarize_pan_records(fit_records, "fit")
pan_dev_summary = summarize_pan_records(dev_records, "dev")
pan_test_summary = summarize_pan_records(test_records, "test")
pan_summary = pd.concat([pan_fit_summary, pan_dev_summary, pan_test_summary], ignore_index=True)

display(
    pan_summary.groupby(["split", "span_type"]).agg(
        n=("pair_id", "size"),
        mean_this_length=("this_length", "mean"),
        median_this_length=("this_length", "median"),
    ).round(2)
)


display(
    pd.DataFrame({
        "split": ["fit", "dev", "test"],
        "pairs": [len(fit_records), len(dev_records), len(test_records)],
        "pairs_with_plagiarism": [
            sum(int(len(r["plagiarism_spans"]) > 0) for r in fit_records),
            sum(int(len(r["plagiarism_spans"]) > 0) for r in dev_records),
            sum(int(len(r["plagiarism_spans"]) > 0) for r in test_records),
        ],
        "pairs_with_altered": [
            sum(int(len(r["altered_spans"]) > 0) for r in fit_records),
            sum(int(len(r["altered_spans"]) > 0) for r in dev_records),
            sum(int(len(r["altered_spans"]) > 0) for r in test_records),
        ],
    })
)


n  mean_this_length  median_this_length
split span_type                                                
dev   altered       87901            706.27               573.0
      plagiarism   188379            781.50               683.0
fit   altered      764449            714.92               585.0
      plagiarism  1689371            785.83               686.0
test  altered      110387            723.26               594.0
      plagiarism   238242            790.29               693.0

,split,pairs,pairs_with_plagiarism,pairs_with_altered
0,fit,55944,38839,29532
1,dev,6216,4301,3305
2,test,7976,5538,4271


Offset-safe normalization

In [ ]:

def normalize_text_with_mapping(
    text: str,
    lowercase: bool = False,
    remove_punctuation: bool = False,
    collapse_whitespace: bool = True,
):
    norm_chars = []
    norm_to_raw = []

    prev_is_space = False
    for raw_idx, ch in enumerate(text):
        out = ch

        if collapse_whitespace and ch.isspace():
            if prev_is_space:
                continue
            out = " "
            prev_is_space = True
        else:
            prev_is_space = False

        if lowercase:
            out = out.lower()

        if remove_punctuation and unicodedata.category(out).startswith("P"):
            continue

        norm_chars.append(out)
        norm_to_raw.append(raw_idx)

    return {
        "norm_text": "".join(norm_chars),
        "norm_to_raw": norm_to_raw,
    }


tokenizer = AutoTokenizer.from_pretrained(CFG.tokenizer_name, use_fast=True)
print("Tokenizer loaded:", CFG.tokenizer_name)
print('Tokenizer fast implementation:', tokenizer.is_fast)


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded: sentence-transformers/all-MiniLM-L6-v2
Tokenizer fast implementation: True


Chunking with exact offsets

In [ ]:

def chunk_document_with_offsets(
    text: str,
    pair_id: str,
    doc_name: str,
    doc_role: str,
    tokenizer,
    cfg: ExperimentConfig,
) -> list[dict]:
    norm_pack = normalize_text_with_mapping(
        text,
        lowercase=cfg.lowercase,
        remove_punctuation=cfg.remove_punctuation,
        collapse_whitespace=cfg.collapse_whitespace,
    )
    norm_text = norm_pack["norm_text"]
    norm_to_raw = norm_pack["norm_to_raw"]

    enc = tokenizer(
        norm_text,
        return_offsets_mapping=True,
        add_special_tokens=False,
        truncation=False,
    )
    offsets = enc["offset_mapping"]

    if len(offsets) == 0:
        return []

    chunks = []
    for start_tok in range(0, len(offsets), cfg.chunk_stride):
        end_tok = min(start_tok + cfg.chunk_tokens, len(offsets))
        if end_tok <= start_tok:
            continue

        start_char_norm = offsets[start_tok][0]
        end_char_norm = offsets[end_tok - 1][1]
        if end_char_norm <= start_char_norm:
            continue

        raw_start = norm_to_raw[start_char_norm]
        raw_end = norm_to_raw[end_char_norm - 1] + 1

        chunk_text = text[raw_start:raw_end]
        chunks.append({
            "pair_id": pair_id,
            "doc_name": doc_name,
            "doc_role": doc_role,
            "chunk_id": f"{pair_id}::{doc_role}::{len(chunks):05d}",
            "token_start": start_tok,
            "token_end": end_tok,
            "norm_char_start": start_char_norm,
            "norm_char_end": end_char_norm,
            "raw_char_start": raw_start,
            "raw_char_end": raw_end,
            "text": chunk_text,
        })

        if end_tok == len(offsets):
            break

    return chunks


def build_pair_chunks(record: dict, tokenizer, cfg: ExperimentConfig):
    susp_chunks = chunk_document_with_offsets(
        text=record["susp_text"],
        pair_id=record["pair_id"],
        doc_name=record["susp_name"],
        doc_role="susp",
        tokenizer=tokenizer,
        cfg=cfg,
    )
    src_chunks = chunk_document_with_offsets(
        text=record["src_text"],
        pair_id=record["pair_id"],
        doc_name=record["src_name"],
        doc_role="src",
        tokenizer=tokenizer,
        cfg=cfg,
    )
    return susp_chunks, src_chunks


example_susp_chunks, example_src_chunks = build_pair_chunks(fit_records[0], tokenizer, CFG)
print("Example pair:", fit_records[0]["pair_id"])
print("Suspicious chunks:", len(example_susp_chunks), "Source chunks:", len(example_src_chunks))
display(pd.DataFrame(example_susp_chunks[:3]))


Token indices sequence length is longer than the specified maximum sequence length for this model (5488 > 512). Running this sequence through the model will result in indexing errors


Example pair: suspicious-document020468__source-document020468
Suspicious chunks: 57 Source chunks: 221


,pair_id,doc_name,doc_role,chunk_id,token_start,token_end,norm_char_start,norm_char_end,raw_char_start,raw_char_end,text
0,suspicious-document020468__source-document020468,suspicious-document020468.txt,susp,suspicious-document020468__source-document0204...,0,192,0,1096,0,1099,Quest for HI Turbulence Statistics: New Techni...
1,suspicious-document020468__source-document020468,suspicious-document020468.txt,susp,suspicious-document020468__source-document0204...,96,288,544,1657,547,1662,on those that can reveal the underlying power ...
2,suspicious-document020468__source-document020468,suspicious-document020468.txt,susp,suspicious-document020468__source-document0204...,192,384,1096,2127,1099,2132,"ynamic turbulence, the cascade from several-kp..."
